# Tech Challenge Fase 2  
## Notebook 07 — FinOps

### Responsabilidade do notebook

Este notebook aplica práticas de **FinOps** à plataforma de dados.

O objetivo é transformar métricas técnicas em indicadores de custo, eficiência e oportunidades de otimização.

A análise contempla:

- volume armazenado por camada;
- quantidade de arquivos;
- estimativa de custo de armazenamento;
- estimativa de custo de leitura;
- estimativa de custo de processamento;
- identificação de arquivos pequenos;
- crescimento por camada;
- eficiência de particionamento;
- recomendações de otimização;
- geração de indicadores para Power BI;
- persistência de histórico.

---

### Importante

Os valores de custo utilizados neste notebook são **parâmetros configuráveis**.

Eles não representam necessariamente a fatura real da AWS ou do Databricks.

O objetivo é demonstrar a metodologia FinOps e permitir que os valores sejam atualizados conforme:

- região AWS;
- classe de armazenamento;
- contrato Databricks;
- tipo de compute;
- quantidade de DBUs;
- política comercial vigente.

---

### Entradas

```text
config/config.json
logs/monitoring/pipeline_inventory
logs/monitoring/storage_metrics
logs/monitoring/platform_status
logs/data_quality/dashboard/kpis
```

### Saídas

```text
logs/finops/storage_cost
logs/finops/processing_cost
logs/finops/efficiency_metrics
logs/finops/recommendations
logs/finops/summary
logs/finops/history
gold/exports_powerbi/finops_dashboard
```

## 1. Contexto FinOps

FinOps combina:

- Engenharia;
- Finanças;
- Operações;
- Governança.

O objetivo não é apenas reduzir custos.

O objetivo é melhorar a relação entre:

```text
Custo
  +
Performance
  +
Qualidade
  +
Valor de negócio
```

Neste projeto, FinOps será aplicado sobre:

- armazenamento no S3;
- organização de arquivos;
- leitura de dados;
- processamento;
- qualidade;
- eficiência da arquitetura Medalhão.

## 2. Importação das bibliotecas

In [0]:
import json
from datetime import datetime

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    LongType,
    DoubleType,
    BooleanType
)

## 3. Leitura das configurações oficiais

Os caminhos são obtidos diretamente do `config.json`.

In [0]:
CONFIG_FILE_PATH = "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase2/config/config.json"

config = json.loads(
    dbutils.fs.head(CONFIG_FILE_PATH)
)

LOG_PATH = config["paths"]["log_path"]
GOLD_PATH = config["paths"]["gold_path"]
CONFIG_PATH = config["paths"]["config_path"]
EXECUTION_DATE = config["project"]["execution_date"]

MONITORING_ROOT_PATH = f"{LOG_PATH}/monitoring"
FINOPS_ROOT_PATH = f"{LOG_PATH}/finops"

FINOPS_STORAGE_PATH = (
    f"{FINOPS_ROOT_PATH}/storage_cost"
)

FINOPS_PROCESSING_PATH = (
    f"{FINOPS_ROOT_PATH}/processing_cost"
)

FINOPS_EFFICIENCY_PATH = (
    f"{FINOPS_ROOT_PATH}/efficiency_metrics"
)

FINOPS_RECOMMENDATIONS_PATH = (
    f"{FINOPS_ROOT_PATH}/recommendations"
)

FINOPS_SUMMARY_PATH = (
    f"{FINOPS_ROOT_PATH}/summary"
)

FINOPS_HISTORY_PATH = (
    f"{FINOPS_ROOT_PATH}/history"
)

POWERBI_FINOPS_PATH = (
    f"{GOLD_PATH}/exports_powerbi/"
    f"finops_dashboard"
)

print("FINOPS_ROOT_PATH:", FINOPS_ROOT_PATH)
print("EXECUTION_DATE:", EXECUTION_DATE)

## 4. Criação dos diretórios de saída

In [0]:
for path in [
    FINOPS_ROOT_PATH,
    FINOPS_STORAGE_PATH,
    FINOPS_PROCESSING_PATH,
    FINOPS_EFFICIENCY_PATH,
    FINOPS_RECOMMENDATIONS_PATH,
    FINOPS_SUMMARY_PATH,
    FINOPS_HISTORY_PATH,
    POWERBI_FINOPS_PATH
]:
    dbutils.fs.mkdirs(path)

print("Diretórios FinOps criados/validados.")

## 5. Parâmetros de custo

Os valores abaixo devem ser tratados como parâmetros de simulação.

Eles podem ser ajustados futuramente sem alterar a lógica do notebook.

In [0]:
finops_parameters = {
    "storage_cost_usd_per_gb_month": 0.023,
    "read_cost_usd_per_gb": 0.005,
    "write_cost_usd_per_gb": 0.005,
    "compute_cost_usd_per_dbu": 0.20,
    "estimated_dbu_per_execution": 1.50,
    "usd_to_brl": 5.50,
    "small_file_threshold_mb": 16.0,
    "target_file_size_mb": 128.0,
    "quality_penalty_cost_usd_per_invalid_record_million": 1.00
}

print(
    json.dumps(
        finops_parameters,
        indent=4,
        ensure_ascii=False
    )
)

## 6. Persistência dos parâmetros FinOps

Os parâmetros são armazenados para rastreabilidade e auditoria.

In [0]:
finops_parameters_path = (
    f"{CONFIG_PATH}/finops_parameters.json"
)

dbutils.fs.put(
    finops_parameters_path,
    json.dumps(
        finops_parameters,
        indent=4,
        ensure_ascii=False
    ),
    overwrite=True
)

print("Parâmetros FinOps salvos em:")
print(finops_parameters_path)

## 7. Funções auxiliares

As funções abaixo realizam:

- leitura segura de Parquet;
- verificação de existência;
- normalização de custos;
- conversão de dólares para reais.

In [0]:
def path_exists(path: str) -> bool:
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False


def read_required_parquet(
    path: str,
    description: str
):
    if not path_exists(path):
        raise FileNotFoundError(
            f"{description} não encontrado: "
            f"{path}"
        )

    return spark.read.parquet(path)


def usd_to_brl(
    value_column
):
    return (
        value_column
        * F.lit(
            finops_parameters[
                "usd_to_brl"
            ]
        )
    )

## 8. Leitura das métricas de Monitoring

O notebook utiliza as métricas reais coletadas pelo `06_monitoring`.

In [0]:
inventory_path = (
    f"{MONITORING_ROOT_PATH}/"
    f"pipeline_inventory/"
    f"execution_date={EXECUTION_DATE}"
)

storage_metrics_path = (
    f"{MONITORING_ROOT_PATH}/"
    f"storage_metrics/"
    f"execution_date={EXECUTION_DATE}"
)

platform_status_path = (
    f"{MONITORING_ROOT_PATH}/"
    f"platform_status/"
    f"execution_date={EXECUTION_DATE}"
)

df_inventory = read_required_parquet(
    inventory_path,
    "Inventário do pipeline"
)

df_storage_metrics = read_required_parquet(
    storage_metrics_path,
    "Métricas de armazenamento"
)

df_platform_status = read_required_parquet(
    platform_status_path,
    "Status da plataforma"
)

display(
    df_inventory
    .orderBy(
        "layer",
        "dataset",
        "ano"
    )
)

## 9. Estimativa de custo de armazenamento

O custo mensal de armazenamento é estimado por:

```text
volume_gb × custo_usd_por_gb_mês
```

In [0]:
df_storage_cost = (
    df_inventory
    .withColumn(
        "estimated_storage_cost_usd_month",
        F.round(
            F.col("size_gb")
            * F.lit(
                finops_parameters[
                    "storage_cost_usd_per_gb_month"
                ]
            ),
            6
        )
    )
    .withColumn(
        "estimated_storage_cost_brl_month",
        F.round(
            usd_to_brl(
                F.col(
                    "estimated_storage_cost_usd_month"
                )
            ),
            6
        )
    )
    .withColumn(
        "execution_date",
        F.lit(EXECUTION_DATE)
    )
)

display(
    df_storage_cost
    .orderBy(
        F.col(
            "estimated_storage_cost_usd_month"
        ).desc()
    )
)

## 10. Estimativa de custo de leitura e escrita

A estimativa considera o volume atual como referência para uma leitura e uma escrita completas por execução.

In [0]:
df_processing_cost = (
    df_inventory
    .withColumn(
        "estimated_read_cost_usd",
        F.round(
            F.col("size_gb")
            * F.lit(
                finops_parameters[
                    "read_cost_usd_per_gb"
                ]
            ),
            6
        )
    )
    .withColumn(
        "estimated_write_cost_usd",
        F.round(
            F.col("size_gb")
            * F.lit(
                finops_parameters[
                    "write_cost_usd_per_gb"
                ]
            ),
            6
        )
    )
    .withColumn(
        "estimated_compute_cost_usd",
        F.round(
            F.lit(
                finops_parameters[
                    "estimated_dbu_per_execution"
                ]
            )
            * F.lit(
                finops_parameters[
                    "compute_cost_usd_per_dbu"
                ]
            ),
            6
        )
    )
    .withColumn(
        "estimated_total_processing_cost_usd",
        F.round(
            F.col(
                "estimated_read_cost_usd"
            )
            + F.col(
                "estimated_write_cost_usd"
            )
            + F.col(
                "estimated_compute_cost_usd"
            ),
            6
        )
    )
    .withColumn(
        "estimated_total_processing_cost_brl",
        F.round(
            usd_to_brl(
                F.col(
                    "estimated_total_processing_cost_usd"
                )
            ),
            6
        )
    )
    .withColumn(
        "execution_date",
        F.lit(EXECUTION_DATE)
    )
)

display(
    df_processing_cost
    .orderBy(
        F.col(
            "estimated_total_processing_cost_usd"
        ).desc()
    )
)

## 11. Análise de arquivos pequenos

Arquivos muito pequenos podem aumentar:

- tempo de listagem;
- custo de leitura;
- overhead de processamento;
- quantidade de tarefas Spark;
- latência.

A análise compara o tamanho médio dos arquivos com o limite configurado.

In [0]:
df_efficiency_metrics = (
    df_inventory
    .withColumn(
        "average_file_size_mb",
        F.when(
            F.col("file_count") > 0,
            F.round(
                F.col("size_mb")
                / F.col("file_count"),
                4
            )
        ).otherwise(
            F.lit(0.0)
        )
    )
    .withColumn(
        "small_file_risk",
        F.when(
            (
                F.col("file_count") > 1
            )
            & (
                F.col(
                    "average_file_size_mb"
                )
                < F.lit(
                    finops_parameters[
                        "small_file_threshold_mb"
                    ]
                )
            ),
            True
        ).otherwise(False)
    )
    .withColumn(
        "estimated_optimal_file_count",
        F.when(
            F.col("size_mb") > 0,
            F.ceil(
                F.col("size_mb")
                / F.lit(
                    finops_parameters[
                        "target_file_size_mb"
                    ]
                )
            )
        ).otherwise(
            F.lit(0)
        )
    )
    .withColumn(
        "file_count_excess",
        F.greatest(
            F.col("file_count")
            - F.col(
                "estimated_optimal_file_count"
            ),
            F.lit(0)
        )
    )
    .withColumn(
        "execution_date",
        F.lit(EXECUTION_DATE)
    )
)

display(
    df_efficiency_metrics
    .orderBy(
        F.col("small_file_risk").desc(),
        F.col("file_count_excess").desc()
    )
)

## 12. Estimativa de penalidade operacional da baixa qualidade

A baixa qualidade pode gerar:

- reprocessamento;
- investigação manual;
- retrabalho;
- atraso na entrega;
- maior uso de compute.

Esta estimativa utiliza a quantidade de registros inválidos como proxy.

In [0]:
quality_kpis_path = (
    f"{LOG_PATH}/data_quality/dashboard/"
    f"kpis/execution_date={EXECUTION_DATE}"
)

df_quality_kpis = read_required_parquet(
    quality_kpis_path,
    "KPIs de qualidade"
)

df_quality_finops = (
    df_quality_kpis
    .withColumn(
        "invalid_records_million",
        F.col("total_invalid_records")
        / F.lit(1_000_000.0)
    )
    .withColumn(
        "estimated_quality_rework_cost_usd",
        F.round(
            F.col(
                "invalid_records_million"
            )
            * F.lit(
                finops_parameters[
                    "quality_penalty_cost_usd_per_invalid_record_million"
                ]
            ),
            6
        )
    )
    .withColumn(
        "estimated_quality_rework_cost_brl",
        F.round(
            usd_to_brl(
                F.col(
                    "estimated_quality_rework_cost_usd"
                )
            ),
            6
        )
    )
)

display(df_quality_finops)

## 13. Recomendações automáticas

O notebook gera recomendações com base em regras FinOps.

In [0]:
recommendations = []

for row in (
    df_efficiency_metrics
    .collect()
):

    if row["small_file_risk"]:

        recommendations.append({
            "layer": row["layer"],
            "dataset": row["dataset"],
            "ano": int(row["ano"]),
            "recommendation_type": (
                "SMALL_FILES"
            ),
            "priority": "HIGH",
            "current_value": float(
                row[
                    "average_file_size_mb"
                ]
            ),
            "target_value": float(
                finops_parameters[
                    "target_file_size_mb"
                ]
            ),
            "recommendation": (
                "Compactar arquivos e reduzir "
                "a quantidade de objetos pequenos."
            ),
            "execution_date": str(
                EXECUTION_DATE
            )
        })

    if row["file_count_excess"] > 0:

        recommendations.append({
            "layer": row["layer"],
            "dataset": row["dataset"],
            "ano": int(row["ano"]),
            "recommendation_type": (
                "FILE_COUNT_EXCESS"
            ),
            "priority": "MEDIUM",
            "current_value": float(
                row["file_count"]
            ),
            "target_value": float(
                row[
                    "estimated_optimal_file_count"
                ]
            ),
            "recommendation": (
                "Revisar particionamento e "
                "aplicar coalesce ou compactação."
            ),
            "execution_date": str(
                EXECUTION_DATE
            )
        })

    if (
        row["component_status"]
        != "DISPONIVEL"
    ):

        recommendations.append({
            "layer": row["layer"],
            "dataset": row["dataset"],
            "ano": int(row["ano"]),
            "recommendation_type": (
                "UNAVAILABLE_COMPONENT"
            ),
            "priority": "CRITICAL",
            "current_value": 0.0,
            "target_value": 1.0,
            "recommendation": (
                "Restaurar a disponibilidade "
                "do componente antes da próxima execução."
            ),
            "execution_date": str(
                EXECUTION_DATE
            )
        })

print(
    "Recomendações geradas:",
    len(recommendations)
)

## 14. Criação do DataFrame de recomendações

In [0]:
schema_recommendations = StructType([
    StructField(
        "layer",
        StringType(),
        False
    ),
    StructField(
        "dataset",
        StringType(),
        False
    ),
    StructField(
        "ano",
        IntegerType(),
        False
    ),
    StructField(
        "recommendation_type",
        StringType(),
        False
    ),
    StructField(
        "priority",
        StringType(),
        False
    ),
    StructField(
        "current_value",
        DoubleType(),
        False
    ),
    StructField(
        "target_value",
        DoubleType(),
        False
    ),
    StructField(
        "recommendation",
        StringType(),
        False
    ),
    StructField(
        "execution_date",
        StringType(),
        False
    )
])

df_finops_recommendations = (
    spark.createDataFrame(
        recommendations,
        schema=schema_recommendations
    )
)

if df_finops_recommendations.count() == 0:
    print("Nenhuma recomendação FinOps necessária.")
else:
    display(
        df_finops_recommendations
        .orderBy(
            F.when(
                F.col("priority") == "CRITICAL",
                1
            )
            .when(
                F.col("priority") == "HIGH",
                2
            )
            .otherwise(3),
            "layer",
            "dataset",
            "ano"
        )
    )

## 15. Resumo executivo FinOps

O resumo consolida custos estimados e eficiência.

In [0]:
df_finops_summary = (
    df_storage_cost
    .agg(
        F.sum(
            "estimated_storage_cost_usd_month"
        ).alias(
            "storage_cost_usd_month"
        ),
        F.sum(
            "estimated_storage_cost_brl_month"
        ).alias(
            "storage_cost_brl_month"
        ),
        F.sum("size_gb").alias(
            "total_storage_gb"
        ),
        F.sum("file_count").alias(
            "total_files"
        )
    )
    .crossJoin(
        df_processing_cost
        .agg(
            F.sum(
                "estimated_total_processing_cost_usd"
            ).alias(
                "processing_cost_usd_execution"
            ),
            F.sum(
                "estimated_total_processing_cost_brl"
            ).alias(
                "processing_cost_brl_execution"
            )
        )
    )
    .crossJoin(
        df_efficiency_metrics
        .agg(
            F.sum(
                F.when(
                    F.col("small_file_risk"),
                    1
                ).otherwise(0)
            ).alias(
                "components_with_small_file_risk"
            ),
            F.sum(
                "file_count_excess"
            ).alias(
                "estimated_excess_files"
            )
        )
    )
    .crossJoin(
        df_quality_finops
        .select(
            "quality_score_percent",
            "estimated_quality_rework_cost_usd",
            "estimated_quality_rework_cost_brl"
        )
    )
    .withColumn(
        "estimated_total_cost_usd",
        F.round(
            F.col(
                "storage_cost_usd_month"
            )
            + F.col(
                "processing_cost_usd_execution"
            )
            + F.col(
                "estimated_quality_rework_cost_usd"
            ),
            6
        )
    )
    .withColumn(
        "estimated_total_cost_brl",
        F.round(
            F.col(
                "storage_cost_brl_month"
            )
            + F.col(
                "processing_cost_brl_execution"
            )
            + F.col(
                "estimated_quality_rework_cost_brl"
            ),
            6
        )
    )
    .withColumn(
        "execution_date",
        F.lit(EXECUTION_DATE)
    )
)

display(df_finops_summary)

## 16. Persistência dos resultados FinOps

In [0]:
outputs = [
    (
        df_storage_cost,
        f"{FINOPS_STORAGE_PATH}/execution_date={EXECUTION_DATE}"
    ),
    (
        df_processing_cost,
        f"{FINOPS_PROCESSING_PATH}/execution_date={EXECUTION_DATE}"
    ),
    (
        df_efficiency_metrics,
        f"{FINOPS_EFFICIENCY_PATH}/execution_date={EXECUTION_DATE}"
    ),
    (
        df_finops_recommendations,
        f"{FINOPS_RECOMMENDATIONS_PATH}/execution_date={EXECUTION_DATE}"
    ),
    (
        df_finops_summary,
        f"{FINOPS_SUMMARY_PATH}/execution_date={EXECUTION_DATE}"
    )
]

for df, path in outputs:

    (
        df
        .coalesce(1)
        .write
        .mode("overwrite")
        .format("parquet")
        .option(
            "compression",
            "snappy"
        )
        .save(path)
    )

print("Resultados FinOps persistidos.")

## 17. Atualização do histórico

In [0]:
(
    df_finops_summary
    .withColumn(
        "snapshot_timestamp",
        F.current_timestamp()
    )
    .write
    .mode("append")
    .format("parquet")
    .partitionBy(
        "execution_date"
    )
    .save(
        FINOPS_HISTORY_PATH
    )
)

print(
    "Histórico FinOps atualizado em:",
    FINOPS_HISTORY_PATH
)

## 18. Exportação para Power BI

In [0]:
df_finops_powerbi = (
    df_efficiency_metrics
    .join(
        df_storage_cost
        .select(
            "layer",
            "dataset",
            "ano",
            "estimated_storage_cost_usd_month",
            "estimated_storage_cost_brl_month"
        ),
        on=[
            "layer",
            "dataset",
            "ano"
        ],
        how="left"
    )
    .join(
        df_processing_cost
        .select(
            "layer",
            "dataset",
            "ano",
            "estimated_total_processing_cost_usd",
            "estimated_total_processing_cost_brl"
        ),
        on=[
            "layer",
            "dataset",
            "ano"
        ],
        how="left"
    )
)

(
    df_finops_powerbi
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .option("sep", ";")
    .option("encoding", "UTF-8")
    .csv(
        POWERBI_FINOPS_PATH
    )
)

print(
    "Dashboard FinOps exportado para:",
    POWERBI_FINOPS_PATH
)

## 19. Checklist final

O notebook apresenta alertas quando:

- existem componentes indisponíveis;
- existem riscos de arquivos pequenos;
- o volume de arquivos está acima do ideal;
- o score de qualidade está abaixo de 95%.

Esses alertas não interrompem automaticamente o pipeline, pois representam oportunidades de otimização e não necessariamente falhas de dados.

In [0]:
summary_row = (
    df_finops_summary
    .first()
)

small_file_components = int(
    summary_row[
        "components_with_small_file_risk"
    ]
    or 0
)

quality_score = float(
    summary_row[
        "quality_score_percent"
    ]
    or 0.0
)

if small_file_components > 0:

    print(
        "ATENÇÃO: existem",
        small_file_components,
        "componentes com risco de "
        "arquivos pequenos."
    )

if quality_score < 95.0:

    print(
        "ATENÇÃO: score de qualidade "
        "abaixo de 95%."
    )

print("FinOps concluído com sucesso.")

print(
    "Custo total estimado em USD:",
    summary_row[
        "estimated_total_cost_usd"
    ]
)

print(
    "Custo total estimado em BRL:",
    summary_row[
        "estimated_total_cost_brl"
    ]
)

## Resultado esperado

Ao final deste notebook estarão disponíveis:

```text
logs/finops/storage_cost/
logs/finops/processing_cost/
logs/finops/efficiency_metrics/
logs/finops/recommendations/
logs/finops/summary/
logs/finops/history/

gold/exports_powerbi/finops_dashboard/
```

### Próxima etapa

```text
08_ia_modelagem
```